# ODI to Databricks Migration: `TRG_EMP` Load

**Source File:** `TRG_EMP_LOAD.txt`
**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook performs a full load (INSERT) of employee data into the `trg_emp` target table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (F/I)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "0", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "0", "ODI Session Number")

## ETL Parameters

The following parameters control the execution of this ETL process. These are defined as Databricks widgets and can be set when running the notebook.

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
    ${ETL_PROC_WID} AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## MERGE into Target

**SCEN_TASK_NO in {10, 20, 30}:** (These steps represent ODI flow control or no-op SQL, handled implicitly by direct execution of the INSERT statement.)

This section performs the main data loading operation into the target table `trg_emp` from `employees`.

*Note: The original Oracle `INSERT /*+ APPEND PARALLEL */` hint has been removed as it's not applicable in Delta Lake. Delta tables inherently handle parallel writes and append operations efficiently.*

In [ ]:
%sql
INSERT INTO workspace.hr.trg_emp
  (
    EMPLOYEE_ID ,
    FIRST_NAME ,
    LAST_NAME ,
    EMAIL ,
    PHONE_NUMBER ,
    HIRE_DATE ,
    JOB_ID ,
    SALARY ,
    COMMISSION_PCT ,
    MANAGER_ID ,
    DEPARTMENT_ID
  )
SELECT
  EMPLOYEES.EMPLOYEE_ID ,
  EMPLOYEES.FIRST_NAME ,
  EMPLOYEES.LAST_NAME ,
  EMPLOYEES.EMAIL ,
  EMPLOYEES.PHONE_NUMBER ,
  EMPLOYEES.HIRE_DATE ,
  EMPLOYEES.JOB_ID ,
  EMPLOYEES.SALARY ,
  EMPLOYEES.COMMISSION_PCT ,
  EMPLOYEES.MANAGER_ID ,
  EMPLOYEES.DEPARTMENT_ID
FROM
  workspace.hr.employees AS EMPLOYEES;

## Validation

Verify the number of records inserted into the target table.

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** Original Oracle schema `HR` has been mapped to `workspace.hr`. Table names `TRG_EMP` and `EMPLOYEES` have been converted to lowercase: `trg_emp` and `employees`.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint was removed as it is specific to Oracle and not applicable to Databricks Delta Lake. Delta Lake handles append operations and parallel writes efficiently by default.
3.  **Data Types:** The column data types in `workspace.hr.trg_emp` and `workspace.hr.employees` are assumed to be Spark SQL compatible (e.g., `NUMBER` to `BIGINT` or `DECIMAL`, `VARCHAR2` to `STRING`, `DATE` to `TIMESTAMP`). If `trg_emp` does not exist, ensure its DDL uses appropriate Spark data types.
4.  **Widget Parameters:** Standard ETL widgets have been included, although they are not directly used in the provided SQL snippet. This ensures consistency with the overall migration framework.
5.  **SCEN_TASK_NOs:** Placeholder SCEN_TASK_NOs {10}, {20}, {30} are noted in the markdown, indicating they were present in the ODI source but did not correspond to explicit SQL operations in this specific snippet.